# Class Imbalance
When one class is rare, accuracy is misleading and models favour the majority.

1. Create an imbalanced dataset and see why accuracy fails
2. Metrics from scratch: precision, recall, F1, ROC-AUC, PR-AUC
3. Logistic regression from scratch (supports sample weights)
4. Fixes: random oversampling, random undersampling, SMOTE, class weights,
   threshold tuning
5. Compare everything on the same test set

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(7)

## 1. An imbalanced dataset
5% positives ("fraud"/"cancer"), two overlapping Gaussian clusters in 2D.

In [ ]:
def make_imbalanced(n=4000, pos_frac=0.05, rng=rng):
    n_pos = int(n * pos_frac); n_neg = n - n_pos
    Xn = rng.normal([0, 0], 1.3, (n_neg, 2))
    Xp = rng.normal([2.0, 2.0], 0.9, (n_pos, 2))
    X = np.vstack([Xn, Xp]); y = np.r_[np.zeros(n_neg), np.ones(n_pos)].astype(int)
    p = rng.permutation(n); return X[p], y[p]

def stratified_split(X, y, test_size=0.3, rng=rng):
    """Keep the class ratio identical in train and test."""
    tr, te = [], []
    for c in np.unique(y):
        idx = rng.permutation(np.where(y == c)[0]); k = int(len(idx) * test_size)
        te += list(idx[:k]); tr += list(idx[k:])
    tr, te = rng.permutation(tr), rng.permutation(te)
    return X[tr], X[te], y[tr], y[te]

X, y = make_imbalanced()
Xtr, Xte, ytr, yte = stratified_split(X, y)
print("Train class counts:", np.bincount(ytr), "| Test:", np.bincount(yte))

## 2. Metrics from scratch
precision = TP/(TP+FP), recall = TP/(TP+FN), F1 = harmonic mean.
ROC-AUC / PR-AUC evaluate the *ranking* of scores over all thresholds.

In [ ]:
def confusion(y, p):
    tp = int(np.sum((y == 1) & (p == 1))); fp = int(np.sum((y == 0) & (p == 1)))
    fn = int(np.sum((y == 1) & (p == 0))); tn = int(np.sum((y == 0) & (p == 0)))
    return tp, fp, fn, tn

def prf(y, p):
    tp, fp, fn, tn = confusion(y, p)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return prec, rec, f1

def accuracy(y, p): return np.mean(y == p)

def pr_curve(y, s):
    o = np.argsort(-s); ys = y[o]
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    return tp / (tp + fp), tp / ys.sum(), tp, fp

def pr_auc(y, s):
    """Average precision: sum over (R_k - R_{k-1}) * P_k."""
    P, R, _, _ = pr_curve(y, s)
    return np.sum(np.diff(np.r_[0, R]) * P)

def roc_auc(y, s):
    """Rank-based (Mann-Whitney) AUC."""
    o = np.argsort(s); ranks = np.empty(len(s)); ranks[o] = np.arange(1, len(s) + 1)
    n1 = y.sum(); n0 = len(y) - n1
    return (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

# The "always healthy" model from the doc:
dummy = np.zeros_like(yte)
print("Always-majority: acc=%.3f prec/rec/F1=%s" % (accuracy(yte, dummy), np.round(prf(yte, dummy), 3)))

## 3. Logistic regression from scratch (with sample weights)
Weighted log-loss: L = - sum_i w_i [y log p + (1-y) log(1-p)] / sum_i w_i.
Class weights are just per-sample weights that depend on the class.

In [ ]:
class LogisticRegression:
    def __init__(self, lr=0.5, epochs=800, l2=1e-3):
        self.lr, self.epochs, self.l2 = lr, epochs, l2
    @staticmethod
    def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    def fit(self, X, y, sample_weight=None):
        n, d = X.shape
        w = np.ones(n) if sample_weight is None else np.asarray(sample_weight, float)
        w = w / w.sum()
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-12
        Z = (X - self.mu) / self.sd
        self.w = np.zeros(d); self.b = 0.0
        for _ in range(self.epochs):
            g = w * (self.sigmoid(Z @ self.w + self.b) - y)
            self.w -= self.lr * (Z.T @ g + self.l2 * self.w)
            self.b -= self.lr * g.sum()
        return self
    def predict_proba(self, X): return self.sigmoid(((X - self.mu) / self.sd) @ self.w + self.b)
    def predict(self, X, thr=0.5): return (self.predict_proba(X) >= thr).astype(int)

def class_weights_balanced(y):
    """w_c = n / (n_classes * n_c)  (same rule as sklearn's 'balanced')."""
    cls, cnt = np.unique(y, return_counts=True)
    m = {c: len(y) / (len(cls) * k) for c, k in zip(cls, cnt)}
    return np.array([m[c] for c in y]), m

base = LogisticRegression().fit(Xtr, ytr)
p0 = base.predict(Xte)
print("Baseline LR: acc=%.3f P/R/F1=%s" % (accuracy(yte, p0), np.round(prf(yte, p0), 3)))

## 4. Fixes
### 4a. Random oversampling / undersampling

In [ ]:
def random_oversample(X, y, rng=rng):
    """Duplicate random minority rows until classes are equal."""
    cls, cnt = np.unique(y, return_counts=True); target = cnt.max()
    idx = []
    for c, k in zip(cls, cnt):
        ci = np.where(y == c)[0]
        idx += list(ci) + list(rng.choice(ci, target - k, replace=True))
    idx = rng.permutation(idx); return X[idx], y[idx]

def random_undersample(X, y, rng=rng):
    """Drop random majority rows until classes are equal."""
    cls, cnt = np.unique(y, return_counts=True); target = cnt.min()
    idx = np.concatenate([rng.choice(np.where(y == c)[0], target, replace=False) for c in cls])
    idx = rng.permutation(idx); return X[idx], y[idx]

### 4b. SMOTE
For each synthetic sample: pick a minority point x, pick one of its k nearest
minority neighbours x_nn, and interpolate: x_new = x + u (x_nn - x), u~U(0,1).

In [ ]:
def smote(X, y, k=5, minority=1, ratio=1.0, rng=rng):
    """Return (X_new, y_new) with minority oversampled to ratio*|majority|."""
    Xm = X[y == minority]; n_maj = np.sum(y != minority)
    n_new = int(ratio * n_maj) - len(Xm)
    if n_new <= 0: return X, y
    D = ((Xm[:, None] - Xm[None]) ** 2).sum(2); np.fill_diagonal(D, np.inf)
    nn = np.argsort(D, axis=1)[:, :k]                     # k nearest minority neighbours
    base = rng.integers(0, len(Xm), n_new)
    neigh = nn[base, rng.integers(0, k, n_new)]
    u = rng.random((n_new, 1))
    Xs = Xm[base] + u * (Xm[neigh] - Xm[base])
    return np.vstack([X, Xs]), np.r_[y, np.full(n_new, minority)]

Xs_, ys_ = smote(Xtr, ytr)
print("After SMOTE class counts:", np.bincount(ys_))

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3), sharex=True, sharey=True)
Xo_, yo_ = random_oversample(Xtr, ytr); Xu_, yu_ = random_undersample(Xtr, ytr)
for a, (XX, yy, t) in zip(ax, [(Xtr, ytr, "Original (imbalanced)"),
                              (Xo_, yo_, "Random oversampling (duplicates)"),
                              (Xs_, ys_, "SMOTE (synthetic points)")]):
    a.scatter(*XX[yy == 0].T, s=4, alpha=.3, label="majority")
    a.scatter(*XX[yy == 1].T, s=12, c='r', alpha=.6, label="minority"); a.set_title(t)
ax[0].legend()
plt.tight_layout(); plt.savefig('plots/01_resampling_scatter.png', dpi=110); plt.close()

### 4c. Class weights and 4d. threshold tuning

In [ ]:
sw, wmap = class_weights_balanced(ytr)
print("Balanced class weights:", {int(k): round(float(v), 2) for k, v in wmap.items()})

def best_threshold(y, s):
    """Threshold maximising F1 (should be chosen on validation data!)."""
    ths = np.linspace(0.02, 0.98, 97)
    f = [prf(y, (s >= t).astype(int))[2] for t in ths]
    return ths[int(np.argmax(f))]

## 5. Compare all strategies on the same untouched test set
Resampling is applied to the TRAINING set only (never to the test set).
The tuned threshold is picked on a validation slice of the training data.

In [ ]:
def run(name, model, thr=0.5):
    s = model.predict_proba(Xte); p = (s >= thr).astype(int)
    P, R, F = prf(yte, p)
    return dict(name=name, acc=accuracy(yte, p), P=P, R=R, F1=F, roc=roc_auc(yte, s), pr=pr_auc(yte, s), s=s)

results = []
results.append(run("Baseline", base))
results.append(run("Oversample", LogisticRegression().fit(*random_oversample(Xtr, ytr))))
results.append(run("Undersample", LogisticRegression().fit(*random_undersample(Xtr, ytr))))
results.append(run("SMOTE", LogisticRegression().fit(*smote(Xtr, ytr))))
results.append(run("Class weights", LogisticRegression().fit(Xtr, ytr, sample_weight=sw)))
# threshold tuning: baseline model, threshold chosen on a validation split
Xa, Xv, ya, yv = stratified_split(Xtr, ytr, 0.3)
m_val = LogisticRegression().fit(Xa, ya); thr = best_threshold(yv, m_val.predict_proba(Xv))
results.append(run(f"Threshold {thr:.2f}", base, thr))

print(f"{'method':18s} {'acc':>6s} {'prec':>6s} {'rec':>6s} {'F1':>6s} {'ROC':>6s} {'PR':>6s}")
for r in results:
    print(f"{r['name']:18s} {r['acc']:6.3f} {r['P']:6.3f} {r['R']:6.3f} {r['F1']:6.3f} {r['roc']:6.3f} {r['pr']:6.3f}")

names = [r['name'] for r in results]; xi = np.arange(len(names)); wd = .2
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for j, (k, lab) in enumerate([('acc', 'Accuracy'), ('P', 'Precision'), ('R', 'Recall'), ('F1', 'F1')]):
    ax[0].bar(xi + (j - 1.5) * wd, [r[k] for r in results], wd, label=lab)
ax[0].set_xticks(xi); ax[0].set_xticklabels(names, rotation=20); ax[0].legend(); ax[0].set_title("Accuracy hides the minority class")
for r in results[:5]:
    P, R, _, _ = pr_curve(yte, r['s']); ax[1].plot(R, P, label=r['name'])
ax[1].axhline(yte.mean(), color='k', ls=':', label="no-skill"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision")
ax[1].set_title("Precision-Recall curves"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig('plots/02_method_comparison.png', dpi=110); plt.close()

# decision boundaries
gx, gy = np.meshgrid(np.linspace(-4, 5, 200), np.linspace(-4, 5, 200)); G = np.c_[gx.ravel(), gy.ravel()]
models = {"Baseline": base, "SMOTE": LogisticRegression().fit(Xs_, ys_),
          "Class weights": LogisticRegression().fit(Xtr, ytr, sample_weight=sw)}
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True, sharey=True)
for a, (n_, m) in zip(ax, models.items()):
    a.contourf(gx, gy, m.predict(G).reshape(gx.shape), alpha=.2, levels=[-.5, .5, 1.5], colors=['tab:blue', 'tab:red'])
    a.scatter(*Xte[yte == 0].T, s=4, alpha=.3); a.scatter(*Xte[yte == 1].T, s=12, c='r'); a.set_title(n_)
plt.tight_layout(); plt.savefig('plots/03_decision_boundaries.png', dpi=110); plt.close()

## Takeaways
* Accuracy of the baseline looks high but recall is poor.
* Reweighting / resampling / threshold tuning trade precision for recall.
* Never resample before splitting (that would be data leakage).

In [ ]:
# Optional sklearn sanity check for the metrics
try:
    from sklearn.metrics import f1_score, roc_auc_score
    p = base.predict(Xte)
    print("F1 ours/sklearn:", round(prf(yte, p)[2], 4), round(f1_score(yte, p), 4),
          "| ROC-AUC ours/sklearn:", round(roc_auc(yte, results[0]['s']), 4), round(roc_auc_score(yte, results[0]['s']), 4))
except Exception as e:
    print("sklearn check skipped:", e)